# 05 - Data Cleaning

## Proyecto
Detección de Sitios Web Fraudulentos (Phishing)

## Objetivo de esta notebook
Aplicar limpieza y transformación al dataset crudo, cubriendo:
- tratamiento de valores nulos
- eliminación de duplicados
- verificación de tipos de datos
- análisis de outliers en el contexto ternario
- encoding y escalado

Esta notebook corresponde al backlog item **PB-07: Limpiar y transformar variables**.

## Resumen de hallazgos del EDA (Sprint 1)

Del análisis realizado en `03_eda.ipynb` se identificaron los siguientes puntos clave:

- **Sin valores nulos**: el dataset no presentó valores faltantes en ninguna columna.
- **Duplicados detectados**: se encontraron filas exactamente duplicadas que deben eliminarse.
- **Tipos de datos**: todas las variables son numéricas enteras (int64), con valores restringidos a {-1, 0, 1}.
- **Variable objetivo `Result`**: binaria en la práctica (-1 phishing, 1 legítimo); la clase 0 no aparece en el target.
- **Balance de clases**: aproximadamente 55% phishing (-1) y 45% legítimo (1), moderadamente desbalanceado.
- **Outliers**: por ser variables ternarias codificadas, los valores fuera de IQR son artefactos del esquema de codificación, no errores reales.

## Importación de librerías

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Carga del dataset crudo

In [5]:
df_raw = pd.read_csv("../data/raw/dataset.csv")
print(f"Dimensiones originales: {df_raw.shape}")
df_raw.head()

Dimensiones originales: (11055, 31)


,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,Domain_registeration_length,Favicon,...,popUpWidnow,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report,Result
0,-1,1,1,1,-1,-1,-1,-1,-1,1,...,1,1,-1,-1,-1,-1,1,1,-1,-1
1,1,1,1,1,1,-1,0,1,-1,1,...,1,1,-1,-1,0,-1,1,1,1,-1
2,1,0,1,1,1,-1,-1,-1,-1,1,...,1,1,1,-1,1,-1,1,0,-1,-1
3,1,0,1,1,1,-1,-1,-1,1,1,...,1,1,-1,-1,1,-1,1,-1,1,-1
4,1,0,-1,1,1,-1,1,1,-1,1,...,-1,1,-1,-1,0,-1,1,1,1,1


## Verificación de valores nulos

Se cuantifican los valores faltantes por columna. **Decisión**: si no hay nulos, no se aplica imputación. Si los hubiera, se aplicaría imputación por moda dado que las variables son categóricas ordinales codificadas.

In [6]:
missing = pd.DataFrame({
    "null_count": df_raw.isnull().sum(),
    "null_pct": (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
}).sort_values("null_count", ascending=False)

print(f"Total de valores nulos: {df_raw.isnull().sum().sum()}")
missing[missing["null_count"] > 0]

Total de valores nulos: 0


,null_count,null_pct


**Conclusión**: no se detectaron valores nulos. No se requiere imputación.

## Tratamiento de duplicados

Se eliminan las filas exactamente duplicadas. **Justificación**: los duplicados exactos no aportan información adicional al modelo y pueden inflar métricas si coinciden en los sets de train y test.

In [7]:
n_duplicados = df_raw.duplicated().sum()
print(f"Filas duplicadas encontradas: {n_duplicados}")
print(f"Porcentaje sobre el total: {n_duplicados / len(df_raw) * 100:.2f}%")

Filas duplicadas encontradas: 5206
Porcentaje sobre el total: 47.09%


In [8]:
# Trabajamos sobre una copia para no modificar el original
df = df_raw.copy()
df = df.drop_duplicates().reset_index(drop=True)

print(f"Dimensiones antes de eliminar duplicados: {df_raw.shape}")
print(f"Dimensiones después de eliminar duplicados: {df.shape}")
print(f"Filas eliminadas: {len(df_raw) - len(df)}")

Dimensiones antes de eliminar duplicados: (11055, 31)
Dimensiones después de eliminar duplicados: (5849, 31)
Filas eliminadas: 5206


## Verificación de tipos de datos

Se confirma que todos los tipos son consistentes con el esquema ternario del dataset.

In [9]:
print("Tipos de datos:")
print(df.dtypes.value_counts())
print("\nDetalle por columna:")
df.dtypes

Tipos de datos:
int64    31
Name: count, dtype: int64

Detalle por columna:


having_IP_Address              int64
URL_Length                     int64
Shortining_Service             int64
having_At_Symbol               int64
double_slash_redirecting       int64
Prefix_Suffix                  int64
having_Sub_Domain              int64
SSLfinal_State                 int64
Domain_registeration_length    int64
Favicon                        int64
port                           int64
HTTPS_token                    int64
Request_URL                    int64
URL_of_Anchor                  int64
Links_in_tags                  int64
SFH                            int64
Submitting_to_email            int64
Abnormal_URL                   int64
Redirect                       int64
on_mouseover                   int64
RightClick                     int64
popUpWidnow                    int64
Iframe                         int64
age_of_domain                  int64
DNSRecord                      int64
web_traffic                    int64
Page_Rank                      int64
G

**Conclusión**: todas las variables son numéricas enteras. No se requieren conversiones de tipo.

## Verificación del rango de valores de las variables

Se comprueba que los valores de cada variable pertenecen al conjunto esperado {-1, 0, 1}. Cualquier valor fuera de este rango sería un error de codificación.

In [10]:
valores_esperados = {-1, 0, 1}
columnas_features = [c for c in df.columns if c != "Result"]

anomalias = {}
for col in columnas_features:
    valores_unicos = set(df[col].unique())
    valores_fuera = valores_unicos - valores_esperados
    if valores_fuera:
        anomalias[col] = valores_fuera

if anomalias:
    print("Variables con valores fuera de {-1, 0, 1}:")
    for col, vals in anomalias.items():
        print(f"  {col}: {vals}")
else:
    print("Todas las variables de features tienen únicamente valores en {-1, 0, 1}. No se detectaron anomalías.")

Todas las variables de features tienen únicamente valores en {-1, 0, 1}. No se detectaron anomalías.


## Análisis de outliers

**Decisión**: dado que las variables son variables ternarias codificadas con valores discretos {-1, 0, 1}, la noción clásica de outlier estadístico no aplica. Los valores fuera del IQR detectados en el EDA son consecuencia del esquema de codificación, no errores reales. **No se aplica tratamiento de outliers.**

Se documenta esta decisión para trazabilidad.

In [11]:
# Distribución de valores únicos por variable para confirmar esquema ternario
valor_counts_por_col = {}
for col in columnas_features:
    valor_counts_por_col[col] = df[col].value_counts().sort_index().to_dict()

# Mostramos cuántas variables usan cada combinación de valores
n_binarias = sum(1 for col in columnas_features if set(df[col].unique()) <= {-1, 1})
n_ternarias = sum(1 for col in columnas_features if set(df[col].unique()) == {-1, 0, 1})

print(f"Variables con valores únicamente en {{-1, 1}} (binarias): {n_binarias}")
print(f"Variables con valores en {{-1, 0, 1}} (ternarias completas): {n_ternarias}")

Variables con valores únicamente en {-1, 1} (binarias): 21
Variables con valores en {-1, 0, 1} (ternarias completas): 8


## Encoding

**Decisión**: las variables ya están numéricamente codificadas como enteros {-1, 0, 1}. No se requiere OneHotEncoding ni LabelEncoding porque los algoritmos de scikit-learn aceptan directamente estos valores numéricos. La codificación actual preserva el orden ordinal implícito (sospechoso → neutral → legítimo).

Se documenta esta decisión para su integración en el Pipeline del Sprint 2.

In [12]:
# Confirmamos que el encoding ya es numérico y no requiere transformación adicional
print("Resumen de tipos post-limpieza:")
print(df.dtypes.value_counts())
print("\nTodas las variables son numéricas: encoding no requerido.")

Resumen de tipos post-limpieza:
int64    31
Name: count, dtype: int64

Todas las variables son numéricas: encoding no requerido.


## Escalado

**Decisión**: se aplica `StandardScaler` sobre las variables de features en el Pipeline. Aunque los valores son discretos {-1, 0, 1}, el escalado es beneficioso para algoritmos sensibles a la magnitud como Regresión Logística, SVM y KNN. Los algoritmos basados en árboles no requieren escalado, pero aplicarlo no los perjudica.

El escalado se encapsula en el Pipeline (notebook `08_pipeline.ipynb`) y se aplica exclusivamente sobre el conjunto de entrenamiento para evitar data leakage.

In [13]:
from sklearn.preprocessing import StandardScaler

# Demostración del escalado (se aplicará formalmente dentro del Pipeline)
X_demo = df[columnas_features].copy()
scaler_demo = StandardScaler()
X_scaled_demo = scaler_demo.fit_transform(X_demo)
X_scaled_df = pd.DataFrame(X_scaled_demo, columns=columnas_features)

print("Estadísticas antes del escalado:")
print(X_demo.describe().loc[["mean", "std"]].T.head(5))
print("\nEstadísticas después del escalado:")
print(X_scaled_df.describe().loc[["mean", "std"]].T.head(5))

Estadísticas antes del escalado:
                              mean       std
having_IP_Address         0.132843  0.991222
URL_Length               -0.616003  0.777323
Shortining_Service        0.720294  0.693728
having_At_Symbol          0.588648  0.808459
double_slash_redirecting  0.718242  0.695852

Estadísticas después del escalado:
                                  mean       std
having_IP_Address         3.887394e-17  1.000085
URL_Length                2.186659e-17  1.000085
Shortining_Service       -3.644432e-18  1.000085
having_At_Symbol          6.802940e-17  1.000085
double_slash_redirecting -7.288864e-17  1.000085


## Verificación antes/después de la limpieza

In [14]:
resumen_comparativo = pd.DataFrame({
    "Métrica": ["Filas", "Columnas", "Valores nulos", "Filas duplicadas"],
    "Antes (raw)": [
        df_raw.shape[0],
        df_raw.shape[1],
        int(df_raw.isnull().sum().sum()),
        int(df_raw.duplicated().sum())
    ],
    "Después (limpio)": [
        df.shape[0],
        df.shape[1],
        int(df.isnull().sum().sum()),
        int(df.duplicated().sum())
    ]
})

resumen_comparativo

,Métrica,Antes (raw),Después (limpio)
0,Filas,11055,5849
1,Columnas,31,31
2,Valores nulos,0,0
3,Filas duplicadas,5206,0


## Guardado del dataset intermedio

In [15]:
import os
os.makedirs("../data/interim", exist_ok=True)

df.to_csv("../data/interim/dataset_clean.csv", index=False)
print(f"Dataset intermedio guardado en ../data/interim/dataset_clean.csv")
print(f"Shape final: {df.shape}")

Dataset intermedio guardado en ../data/interim/dataset_clean.csv
Shape final: (5849, 31)


## Resumen de PB-07

En esta notebook se realizaron las siguientes acciones de limpieza y transformación:

- **Valores nulos**: no se detectaron; no se requirió imputación.
- **Duplicados**: se eliminaron las filas duplicadas exactas.
- **Tipos de datos**: verificados y confirmados como numéricos enteros consistentes con el esquema {-1, 0, 1}.
- **Outliers**: no aplicables dado el esquema de codificación ternaria discreta; decisión documentada.
- **Encoding**: no requerido; las variables ya son numéricas.
- **Escalado**: se documentó la decisión de aplicar `StandardScaler` dentro del Pipeline formal.
- **Dataset intermedio**: guardado en `data/interim/dataset_clean.csv`.

Esta notebook corresponde al backlog item **PB-07: Limpiar y transformar variables**.